# H1+ZEUS NC inclusive data → pandas + **per-$Q^2$** baseline + GCFT **xion** prior (illustrative)

Public data: **Aaron et al., JHEP 01 (2010) 109** — combined H1 and ZEUS inclusive NC $e^+p$ on [HEPData ins836107](https://www.hepdata.net/record/ins836107). Tables list **reduced cross section** $\sigma_r$ (with breakdown of `%` uncertainties) and **$F_2$** where quoted.

**Scope:** This submission is **inclusive NC DIS** ($\sigma_r$, $F_2$). Longitudinal $F_L$, diffractive $\sigma^{\mathrm{difr}}$, and other processes live in **other** HepData entries — extend the same loader pattern when you pin record IDs.

**Open in Colab:** [GitHub → Colab](https://colab.research.google.com/) → paste `ChaonickGCFT/Zeus_Notebook` / `notebooks/gcft_hera_f2_xion_prior.ipynb`.

**Xion** = small multiplicative bump in $\log x$ with Gaussian priors — a **scaffold**, not a published GCFT likelihood.

In [ ]:
import io
import re
import tarfile
import gzip
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HEPDATA_CSV_TGZ = "https://www.hepdata.net/download/submission/ins836107/1/csv"

In [ ]:
def _parse_pct(tok: str) -> float:
    tok = tok.strip().strip('"').rstrip("%")
    return float(tok) / 100.0


def load_hepdata_ins836107_bundle(url: str = HEPDATA_CSV_TGZ) -> pd.DataFrame:
    """Parse each Table*.csv: reduced cross section block + $F_2$ block; merge on $(Q^2,x,y,E_{\rm cm})$."""
    raw = urlopen(url).read()
    sig_rows, f2_rows = [], []
    with tarfile.open(fileobj=gzip.GzipFile(fileobj=io.BytesIO(raw)), mode="r|") as tar:
        for member in tar:
            if not member.isfile() or not member.name.endswith(".csv"):
                continue
            text = tar.extractfile(member).read().decode("utf-8", errors="replace")
            q2 = None
            mode = None
            for line in text.splitlines():
                line = line.strip()
                if not line:
                    continue
                m = re.match(r"#:\s*Q\*\*2\s*\[GeV\^2\]\s*,\s*([0-9.eE+-]+)", line)
                if m:
                    q2 = float(m.group(1))
                    mode = None
                    continue
                if line.startswith("#"):
                    continue
                ul = line.upper()
                if ul.startswith("X,") and "SIG" in ul and "REDUCED" in ul.replace(" ", ""):
                    mode = "sig"
                    continue
                if ul.startswith("X,") and "F2" in ul and "SIG" not in ul:
                    mode = "f2"
                    continue
                if mode == "sig" and q2 is not None:
                    parts = [p.strip() for p in line.split(",")]
                    if len(parts) < 6:
                        continue
                    try:
                        x, y, ecm = float(parts[0]), float(parts[1]), float(parts[2])
                        sig = float(parts[3])
                        dp = _parse_pct(parts[4])
                        dm = abs(_parse_pct(parts[5]))
                    except ValueError:
                        continue
                    stat_rel = 0.5 * (dp + dm)
                    sig_rows.append(
                        {
                            "source": member.name,
                            "Q2": q2,
                            "x": x,
                            "y": y,
                            "Ecm": ecm,
                            "sigma_r": sig,
                            "sigma_r_stat_rel": stat_rel,
                        }
                    )
                elif mode == "f2" and q2 is not None:
                    parts = [p.strip() for p in line.split(",")]
                    if len(parts) < 4:
                        continue
                    try:
                        x, y, ecm = float(parts[0]), float(parts[1]), float(parts[2])
                    except ValueError:
                        continue
                    f2tok = parts[3].strip()
                    if f2tok in ("-", "", "nan"):
                        continue
                    try:
                        f2 = float(f2tok)
                    except ValueError:
                        continue
                    norm_rel = 0.005
                    if len(parts) >= 6 and "%" in parts[4]:
                        try:
                            norm_rel = max(norm_rel, abs(_parse_pct(parts[4])))
                        except ValueError:
                            pass
                    f2_rows.append(
                        {
                            "source": member.name,
                            "Q2": q2,
                            "x": x,
                            "y": y,
                            "Ecm": ecm,
                            "F2": f2,
                            "F2_norm_rel": norm_rel,
                        }
                    )
    sig_df = pd.DataFrame(sig_rows)
    f2_df = pd.DataFrame(f2_rows)
    keys = ["Q2", "x", "y", "Ecm"]
    sig_df = sig_df.drop_duplicates(subset=keys, keep="first")
    f2_df = f2_df.drop_duplicates(subset=keys, keep="first")
    merged = f2_df.merge(sig_df[keys + ["sigma_r", "sigma_r_stat_rel"]], on=keys, how="left")
    merged["F2_rel_err"] = np.nan
    m = merged["sigma_r_stat_rel"].notna()
    merged.loc[m, "F2_rel_err"] = merged.loc[m, "sigma_r_stat_rel"]
    merged["F2_rel_err"] = merged["F2_rel_err"].fillna(
        merged["F2_norm_rel"].clip(lower=0.005)
    )
    return merged


df = load_hepdata_ins836107_bundle()
df = df[np.isfinite(df["F2"]) & (df["F2"] > 0)].copy()
df["logx"] = np.log(df["x"])
df["logQ2"] = np.log(df["Q2"])
df["logF2"] = np.log(df["F2"])
print(len(df), "$F_2$ points after merge")
print("$Q^2$ range:", df["Q2"].min(), "…", df["Q2"].max(), "GeV$^2$")
print("Matched $\sigma_r$ rows:", df["sigma_r"].notna().sum(), "/", len(df))
df.head()

## Baselines

1. **Global plane** — $\log F_2 = c_0 + c_1\log x + c_2\log Q^2$ (original toy).
2. **Global quadratic surface** — adds $(\log x)^2$, $(\log Q^2)^2$, $(\log x)(\log Q^2)$ for curvature.
3. **Per-$Q^2$ line** — for each table’s $Q^2$, $\log F_2 = a(Q^2) + b(Q^2)\log x$ (minimum 2 points). If a bin is degenerate, fall back to the quadratic surface.

## Xion prior (same as before)

$$
F_2^{\rm pred} = F_2^{\rm base}\,\exp\bigl(\eta\,\phi(x)\bigr),\quad
\phi(x)=\exp\!\left(-\frac{(\log(x/x_0))^2}{2w^2}\right),
$$
with $\eta\sim\mathcal{N}(0,\sigma_\eta^2)$, $\log_{10}x_0$ Gaussian, fixed $w$.

In [ ]:
def fit_global_plane(d: pd.DataFrame):
    X = np.column_stack([np.ones(len(d)), d["logx"], d["logQ2"]])
    coef, _, _, _ = np.linalg.lstsq(X, d["logF2"], rcond=None)
    return coef


def fit_global_quad(d: pd.DataFrame):
    lx, lq = d["logx"].values, d["logQ2"].values
    X = np.column_stack(
        [np.ones(len(d)), lx, lq, lx ** 2, lq ** 2, lx * lq]
    )
    coef, _, _, _ = np.linalg.lstsq(X, d["logF2"], rcond=None)
    return coef


def log_f2_global_plane(x, Q2, coef):
    c0, c1, c2 = coef
    return c0 + c1 * np.log(x) + c2 * np.log(Q2)


def log_f2_global_quad(x, Q2, coef):
    c0, c1, c2, c3, c4, c5 = coef
    lx = np.log(x)
    lq = np.log(Q2)
    return c0 + c1 * lx + c2 * lq + c3 * lx ** 2 + c4 * lq ** 2 + c5 * lx * lq


def fit_per_q2_lines(d: pd.DataFrame):
    """Return dict Q2 -> (a,b) for log F2 = a + b log x."""
    out = {}
    for q2, g in d.groupby("Q2"):
        if len(g) < 2:
            out[q2] = None
            continue
        X = np.column_stack([np.ones(len(g)), g["logx"].values])
        c, _, _, _ = np.linalg.lstsq(X, g["logF2"].values, rcond=None)
        out[q2] = c
    return out


def log_f2_per_q2(x, Q2, per_q2, coef_quad):
    x = np.asarray(x, dtype=float)
    Q2 = np.asarray(Q2, dtype=float)
    out = np.empty_like(x, dtype=float)
    xr, qr, orv = x.ravel(), Q2.ravel(), out.ravel()
    for i in range(xr.size):
        xi, qi = xr[i], qr[i]
        c = per_q2.get(qi)
        if c is None:
            c = per_q2.get(float(qi))
        if c is not None:
            orv[i] = c[0] + c[1] * np.log(xi)
        else:
            orv[i] = log_f2_global_quad(xi, qi, coef_quad)
    return out


coef_plane = fit_global_plane(df)
coef_quad = fit_global_quad(df)
per_q2 = fit_per_q2_lines(df)


def phi_xion(x, x0, w):
    return np.exp(-0.5 * (np.log(x / x0) / w) ** 2)


def f2_from_log(logf2, eta, phi):
    return np.exp(logf2 + eta * phi)


rng = np.random.default_rng(42)
sigma_eta = 0.04
mu_log10_x0, tau_log10_x0 = -3.0, 0.35
w_xion = 1.15
n_draw = 400

etas = rng.normal(0.0, sigma_eta, size=n_draw)
log10_x0s = rng.normal(mu_log10_x0, tau_log10_x0, size=n_draw)
x0s = 10.0 ** log10_x0s

xv = df["x"].values
Q2v = df["Q2"].values
f2v = df["F2"].values
sig_rel = df["F2_rel_err"].values

log_plane = log_f2_global_plane(xv, Q2v, coef_plane)
log_perq = log_f2_per_q2(xv, Q2v, per_q2, coef_quad)


def residual_samples(log_base):
    cols = []
    for eta, x0 in zip(etas, x0s):
        phi = phi_xion(xv, x0, w_xion)
        fp = f2_from_log(log_base, eta, phi)
        cols.append((f2v - fp) / f2v)
    return np.column_stack(cols)


rs_plane = residual_samples(log_plane)
rs_perq = residual_samples(log_perq)

med_p = np.median(rs_plane, axis=1)
med_q = np.median(rs_perq, axis=1)
pull_p = med_p / sig_rel
pull_q = med_q / sig_rel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharex=True)
for ax, med, title in zip(
    axes,
    [med_p, med_q],
    ["global log-plane baseline", "per-$Q^2$ log-$x$ line (+ quad fallback)"],
):
    sc = ax.scatter(Q2v, med, c=np.log10(xv), s=10, alpha=0.72, cmap="viridis")
    ax.axhline(0.0, color="k", lw=0.8, ls="--")
    ax.set_xscale("log")
    ax.set_xlabel(r"$Q^2\;(GeV^2)$")
    ax.set_ylabel(r"median rel.~res. $(F_2^{data}-F_2^{pred})/F_2^{data}$")
    ax.set_title(title + " + xion prior draws")
    lo, hi = np.nanpercentile(med, [2, 98])
    pad = max(0.05, 0.15 * (hi - lo))
    ax.set_ylim(lo - pad, hi + pad)

fig.suptitle("H1+ZEUS combined NC $F_2$: baseline comparison (xion median over prior)", y=1.02)
fig.colorbar(sc, ax=axes, label=r"$\log_{10}(x)$", fraction=0.02, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(pull_q[np.isfinite(pull_q)], bins=40, color="steelblue", alpha=0.85, edgecolor="white")
ax.axvline(0.0, color="k", ls="--", lw=0.9)
ax.set_xlabel(r"pull (median residual / $\hat\sigma_{F_2}^{\mathrm{proxy}}$)")
ax.set_ylabel("counts")
ax.set_title("Proxy pulls: $\hat\sigma$ = matched $\sigma_r$ stat (else norm 0.5%)")
plt.tight_layout()
plt.show()

In [ ]:
q2_pick = sorted(df["Q2"].unique())
idx = np.linspace(0, len(q2_pick) - 1, 4, dtype=int)
chosen = [q2_pick[i] for i in idx]

fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=False, sharey=False)
axes = axes.ravel()
xx_fine = np.logspace(np.log10(df["x"].min()), np.log10(df["x"].max()), 120)

for ax, q2 in zip(axes, chosen):
    sub = df[np.isclose(df["Q2"], q2)]
    ax.errorbar(
        sub["x"],
        sub["F2"],
        yerr=sub["F2_rel_err"] * sub["F2"],
        fmt="o",
        ms=4,
        capsize=2,
        color="0.2",
        alpha=0.85,
        label="data",
    )
    lb = log_f2_per_q2(xx_fine, np.full_like(xx_fine, q2), per_q2, coef_quad)
    ax.plot(xx_fine, np.exp(lb), color="C1", lw=1.8, label="per-$Q^2$ baseline")
    curves = []
    for eta, x0 in list(zip(etas, x0s))[:80]:
        phi = phi_xion(xx_fine, x0, w_xion)
        curves.append(np.exp(lb + eta * phi))
    arr = np.row_stack(curves)
    ax.fill_between(
        xx_fine,
        np.percentile(arr, 16, axis=0),
        np.percentile(arr, 84, axis=0),
        color="C0",
        alpha=0.25,
        label="xion 68% (subset)",
    )
    ax.set_xscale("log")
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$F_2$")
    ax.set_title(rf"$Q^2 = {q2:g}$ GeV$^2$")
    ax.legend(fontsize=8, loc="best")

fig.suptitle("$F_2$ vs $x$ at four $Q^2$ slices (illustrative xion band)", y=1.01)
plt.tight_layout()
plt.show()

### Quick checklist

- **Per-$Q^2$ baseline** usually shrinks the huge global-plane misfit at the kinematic edges.
- **Proxy $\sigma_{F_2}$** uses the **statistical** `%` from the $\sigma_r$ row at the same $(x,y,Q^2)$; it ignores correlated systematics — use HepData YAML if you need the full covariance.
- **Next data steps** (same loader idea): find HepData records for $F_L$, diffractive reduced cross sections, etc., and merge by DOI / record id.